# PyOccam Demo Notebook

This notebook demonstrates the complete OCCAM workflow using the Python bindings.

## Installation Check

In [1]:
# Check that pyoccam is installed and working
import pyoccam
print(f"PyOccam version: {pyoccam.__version__}")
print(f"Available functions: load_dementia, load_landslides, load_data, quick_search")
print(f"Constants: SPACESEP={pyoccam.SPACESEP}, COMMASEP={pyoccam.COMMASEP}, TABSEP={pyoccam.TABSEP}")

PyOccam 0.1.2 loaded successfully
PyOccam 0.1.2 loaded. Type pyoccam.help() for usage.
PyOccam version: 0.1.2
Available functions: load_dementia, load_landslides, load_data, quick_search
Constants: SPACESEP=3, COMMASEP=2, TABSEP=1


## 1. Load Data

PyOccam provides sklearn-style data loading functions:

In [2]:
# Load the dementia dataset
data = pyoccam.load_dementia()

# Display dataset information
print(f"Dataset: dementia05.txt")
print(f"Samples: {data.n_samples}")
print(f"Features: {data.n_features}")
print(f"Target variable: {data.target_name}")
print(f"Has test data: {data.has_test_data}")
print(f"\nFeature names ({len(data.feature_names)}):")
for i, name in enumerate(data.feature_names[:5]):
    print(f"  {i+1}. {name}")
if len(data.feature_names) > 5:
    print(f"  ... and {len(data.feature_names)-5} more")

✓ Loaded dementia: 424 samples, 18 features
Dataset: dementia05.txt
Samples: 424
Features: 18
Target variable: CaseControl
Has test data: False

Feature names (18):
  1. APOE
  2. Gender
  3. Education
  4. AgeLastExam
  5. rs1801133
  ... and 13 more


## 2. Configure OCCAM Settings

In [3]:
# Get the manager from the data object
manager = data.manager

# Configure output format and reference model
manager.set_report_separator(pyoccam.SPACESEP)  # Space-separated output
manager.set_ref_model("bottom")  # Use bottom reference for up searches

# Set report variables (don't include ID or Model - they're automatic)
manager.set_report_variables("Level$I, h, ddf, dLR, Alpha, Inf, %dH(DV), dAIC, dBIC")

print("✓ OCCAM configured")

✓ OCCAM configured


## 3. Run Search

Search for the best models using beam search:

In [4]:
# Search parameters
SEARCH_TYPE = "loopless-up"  # or "full-up" for more comprehensive search
SEARCH_LEVELS = 5  # How deep to search
SEARCH_WIDTH = 3   # How many models to keep at each level

print(f"Running {SEARCH_TYPE} search...")
print(f"  Levels: {SEARCH_LEVELS}")
print(f"  Width: {SEARCH_WIDTH}")

# Run the search
import time
start = time.time()
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH
)
elapsed = time.time() - start

print(f"\n✓ Search completed in {elapsed:.2f} seconds")
print(f"Models kept: {manager.get_search_model_count()}")

Running loopless-up search...
  Levels: 5
  Width: 3

✓ Search completed in 0.12 seconds
Models kept: 16


## 4. Examine Search Results

In [5]:
# Get the best models
best_bic = manager.get_best_model_by_bic()
best_aic = manager.get_best_model_by_aic()
best_info = manager.get_best_model_by_information()

print("Best models found:")
print(f"  By BIC:         {best_bic}")
print(f"  By AIC:         {best_aic}")
print(f"  By Information: {best_info}")

# Display search report (first 20 lines)
print("\nSearch Report:")
print("="*70)
lines = search_report.split('\n')
for line in lines[:20]:
    print(line)
if len(lines) > 20:
    print(f"... ({len(lines)-20} more lines)")

Best models found:
  By BIC:         IV:ApZ
  By AIC:         IV:ApSxAZ
  By Information: IV:ApSxACEZ

Search Report:
Searching levels:
1 : 18 new models, 3 kept; 4 total kept
2 : 48 new models, 3 kept; 7 total kept
3 : 45 new models, 3 kept; 10 total kept
4 : 42 new models, 3 kept; 13 total kept
5 : 40 new models, 3 kept; 16 total kept

  ID   MODEL                  Level              H            dDF            dLR          Alpha            Inf        %dH(DV)           dAIC           dBIC
  16   IV:ApSxACEZ                5         9.3964            107                        0.0000                       33.0656       -19.8968      -453.2183
  15   IV:ApSxACHZ                5         9.4049            107                        0.0000                       32.2184       -24.8702      -458.1917
  14   IV:ApSxAgAEZ               5         9.4074            107                        0.0000                       31.9605       -26.3839      -459.7054
  13   IV:ApSxAEZ                 4 

## 5. Fit the Best Model

Generate a detailed fit report for the best model:

In [9]:
if best_info:
    print(f"Generating fit report for: {best_info}")
    
    # Generate fit report with confusion matrix for state "0"
    fit_report = manager.generate_fit_report(best_info, "0")
    
    # Display first part of fit report
    print("\nFit Report (first 30 lines):")
    print("="*70)
    fit_lines = fit_report.split('\n')
    for line in fit_lines[:30]:
        print(line)
    if len(fit_lines) > 30:
        print(f"... ({len(fit_lines)-30} more lines)")
    
    # Save the complete report
    with open(f"fit_report_{best_info.replace(':', '_')}.txt", 'w') as f:
        f.write(fit_report)
    print(f"\n✓ Complete fit report saved to file")

Generating fit report for: IV:ApSxACEZ

Fit Report (first 30 lines):
Sample size: 424
Variables: 19

    Model,IV:ApSxACEZ (Directed System)
    IV Component:,APOE; Gender; Education; AgeLastExam; rs1801133; rs3818361; rs7561528; rs744373; rs6943822; rs4298437; rs7012010; rs11136000; rs10786998; rs11193130; rs610932; rs3851179; rs3764650; rs3865444,ApSxEdAgABCDEFGHJKLMNP
    Model Component: ,APOE; Gender; rs1801133; rs7561528; rs6943822; CaseControl,ApSxACEZ
    Degrees of Freedom (DF):,7.25594e+08
    Loops:,NO
    Entropy(H):,9.39639
    Information captured (%):,33.0656
    Transmission (T):,0.668474

-------------------------------------------------------------------------

    REFERENCE = TOP
    ,Value,Prob. (Alpha)
    Log-Likelihood (LR),392.921,1
    Pearson X2,187.123,1
    Delta DF (dDF),7.25594e+08,

-------------------------------------------------------------------------

    REFERENCE = BOTTOM
    ,Value,Prob. (Alpha)
    Log-Likelihood (LR),194.103,0
    Pearson X2,187

## 6. Quick Search Alternative

For quick analysis, you can use the convenience method:

In [ ]:
# Quick search on the data object
best_model = data.quick_search(search_type="loopless-up", levels=3, width=3)
print(f"Quick search found: {best_model}")

## 7. Try Different Datasets

You can also load the landslides dataset or your own data:

In [7]:
# Load landslides dataset
# landslides = pyoccam.load_landslides()
# print(f"Landslides: {landslides.n_samples} samples, {landslides.n_features} features")

# Or load your own data file (must be in OCCAM format)
# custom_data = pyoccam.load_data("mydata.txt")

print("Uncomment the lines above to load different datasets")

Uncomment the lines above to load different datasets


## Summary

This notebook demonstrated:
1. Loading data with sklearn-style functions
2. Configuring OCCAM settings
3. Running model searches
4. Selecting best models by different criteria
5. Generating detailed fit reports
6. Using convenience methods for quick analysis

For more information, see the OCCAM manual or run `pyoccam.help()`